In [14]:
import math
import numpy as np

def _norm_cdf(x):
    """Standard normal CDF using math.erf (no scipy required)."""
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def black_scholes_price(S, K, T, r, sigma, option='call', q=0.0):
    """
    European Black-Scholes price.
    S: spot price
    K: strike
    T: time to maturity (in years)
    r: risk-free rate (annual, continuous)
    sigma: volatility (annual)
    option: 'call' or 'put'
    q: continuous dividend yield (default 0)
    """
    # - Funkcja liczy teoretyczną cenę europejskiej opcji (call lub put)
    #   według modelu Blacka‑Scholesa.
    # - S to bieżąca cena instrumentu (np. akcji), K to cena wykonania (strike).
    # - T to czas do wygaśnięcia w latach (np. 0.5 = 6 miesięcy).
    # - r to bezpieczna stopa procentowa (np. oprocentowanie obligacji),
    #   sigma to oczekiwana zmienność ceny (im wyższa, tym droższa opcja).
    # - q to ciągła stopa dywidendy (jeśli instrument wypłaca dywidendy).
    # - Wynik to ile teoretycznie warto zapłacić dzisiaj za daną opcję.
    # Przykład:
    #   black_scholes_price(100, 100, 1.0, 0.01, 0.2, 'call')
    if T <= 0 or sigma <= 0:
        # immediate payoff if expired or zero vol
        if option.lower().startswith('c'):
            return max(S - K, 0.0)
        return max(K - S, 0.0)

    d1 = (math.log(S / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if option.lower().startswith('c'):
        return S * math.exp(-q * T) * _norm_cdf(d1) - K * math.exp(-r * T) * _norm_cdf(d2)
    return K * math.exp(-r * T) * _norm_cdf(-d2) - S * math.exp(-q * T) * _norm_cdf(-d1)

# Example:
price_call = black_scholes_price(S=100, K=100, T=1.0, r=0.01, sigma=0.2, option='call')
price_call
price_put  = black_scholes_price(S=100, K=100, T=1.0, r=0.01, sigma=0.2, option='put')

In [ ]:
S = 100.0
K = 100.0
T = 1.0
r = 0.01
sigma = 0.2
q = 0.0

# Ustawienia Monte Carlo
np.random.seed(42)
N = 200_000  # liczba symulacji

# Symulacja cen końcowych wg GBM
Z = np.random.normal(size=N)
ST = S * np.exp((r - q - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

# Payoffy i dyskontowanie
call_payoffs = np.maximum(ST - K, 0.0)
put_payoffs = np.maximum(K - ST, 0.0)
discount = np.exp(-r * T)

call_price_mc = discount * call_payoffs.mean()
put_price_mc = discount * put_payoffs.mean()

# Błędy standardowe i przedziały ufności 95%
call_se = discount * call_payoffs.std(ddof=1) / np.sqrt(N)
put_se = discount * put_payoffs.std(ddof=1) / np.sqrt(N)
z = 1.96
call_ci = (call_price_mc - z * call_se, call_price_mc + z * call_se)
put_ci = (put_price_mc - z * put_se, put_price_mc + z * put_se)

# Wyniki porównane z analitycznymi price_call / price_put
print(f"MC call: {call_price_mc:.6f} ± {z*call_se:.6f} (95% CI {call_ci[0]:.6f}, {call_ci[1]:.6f})")
print(f"Analityczny call: {price_call:.6f}  Różnica: {call_price_mc - price_call:.6f}\n")
# print(f"MC put:  {put_price_mc:.6f} ± {z*put_se:.6f} (95% CI {put_ci[0]:.6f}, {put_ci[1]:.6f})")
# print(f"Analityczny put:  {price_put:.6f}  Różnica: {put_price_mc - price_put:.6f}")

MC call: 8.442617 ± 0.059013 (95% CI 8.383604, 8.501629)
Analityczny call: 8.433319  Różnica: 0.009298



# Wyjaśnienie kodu (komórki z funkcją i symulacją Monte Carlo)

Poniżej znajduje się szczegółowe objaśnienie krok po kroku.

## Funkcje i stałe (CELL INDEX: 0)
- _norm_cdf(x)
    - Zwraca dystrybuantę standardowego rozkładu normalnego za pomocą funkcji error function: Φ(x) = 0.5 * (1 + erf(x / √2)).
    - Dzięki temu nie potrzebujemy scipy.

- black_scholes_price(S, K, T, r, sigma, option='call', q=0.0)
    - Oblicza teoretyczną cenę europejskiej opcji według modelu Black‑Scholes (call lub put).
    - Parametry:
        - S: cena spot
        - K: strike
        - T: czas do wygaśnięcia (w latach)
        - r: stopa wolna od ryzyka (ciągłe kapitalizowanie)
        - sigma: zmienność roczna
        - q: ciągły yield/dywidenda
    - Obsługa brzegowa:
        - Jeśli T <= 0 lub sigma <= 0, funkcja zwraca natychmiastowy payoff (max(S-K,0) dla call, max(K-S,0) dla put).
    - Wzory:
        - d1 = (ln(S/K) + (r - q + 0.5*sigma^2) * T) / (sigma * sqrt(T))
        - d2 = d1 - sigma * sqrt(T)
        - Call: S e^{-qT} Φ(d1) − K e^{-rT} Φ(d2)
        - Put: K e^{-rT} Φ(−d2) − S e^{-qT} Φ(−d1)

- Przykład: price_call i price_put obliczone na końcu komórki (użyte później jako odniesienie).

## Symulacja Monte Carlo (CELL INDEX: 1)
Cel: oszacować cenę europejskich opcji przez symulację końcowych cen akcji zgodnie z Geometric Brownian Motion (GBM) i porównać z cenami analitycznymi.

1. Parametry wejściowe
     - S, K, T, r, sigma, q ustawione na typowe wartości (tu S=K=100, T=1, r=0.01, sigma=0.2, q=0).
     - N = 200_000 — liczba ścieżek symulacji.
     - np.random.seed(42) — ustawia ziarno losowości, zapewnia powtarzalność wyników.

2. Symulacja cen końcowych ST (wektorowa operacja numpy)
     - Z = np.random.normal(size=N) — N próbek z N(0,1).
     - Model GBM (wzór analityczny dla ceny w czasie T):
         ST = S * exp((r - q - 0.5 * sigma^2) * T + sigma * sqrt(T) * Z)
     - To jest bezpośrednia symulacja log‑normalnego rozkładu wynikającego z modelu Blacka‑Scholesa.

3. Obliczanie payoffów
     - call_payoffs = np.maximum(ST - K, 0.0)
     - put_payoffs  = np.maximum(K - ST, 0.0)
     - Payoffy są wektorami o długości N.

4. Dyskontowanie do wartości bieżącej
     - discount = exp(-r * T)
     - Szacowane ceny:
         - call_price_mc = discount * mean(call_payoffs)
         - put_price_mc  = discount * mean(put_payoffs)

5. Estymacja niepewności (błąd standardowy i 95% CI)
     - Standard error (dla ceny z dyskontowaniem): se = discount * std(sample, ddof=1) / sqrt(N)
         - ddof=1 używa niewyrownanej estymacji odchylenia standardowego (sample std).
     - CI 95%: (estimate − z*se, estimate + z*se) z z = 1.96.

6. Porównanie z wartością analityczną
     - Drukowany wynik porównuje call_price_mc z price_call (analityczny obliczony przez black_scholes_price).
     - W przykładzie uzyskane wartości to:
         - call_price_mc ≈ 8.442616622574866
         - price_call (analityczny) = 8.433318690109608
         - 95% CI dla call: (≈ 8.38360411, 8.50162914)
         - Różnica ≈ 0.0093 (czyli wynik symulacji jest bliski wartości analitycznej, zmienność zgodna z CI).

## Uwagi praktyczne
- Symulacja jest wektorowa (numpy) → szybka przy dużych N.
- Ustawienie ziarna (seed) daje powtarzalność wyników.
- N=200k daje stosunkowo mały błąd standardowy; można zmniejszyć błąd zwiększając N lub stosując techniki redukcji wariancji (control variates, antithetic variates).
- Dla T <= 0 lub sigma <= 0 funkcja zwraca bezpośredni payoff — ważne zabezpieczenie przed dzieleniem przez zero.
- Porównanie MC z analitycznym jest użytecznym testem poprawności implementacji.

Jeśli chcesz, mogę:
- dodać wariant z redukcją wariancji (np. antithetic),
- policzyć grecki (delta, vega) numerycznie z tej samej symulacji,
- narysować histogram ST lub rozkład payoffów.

In [13]:
print(f"MC call: {call_price_mc:.6f} ± {z*call_se:.6f} (95% CI {call_ci[0]:.6f}, {call_ci[1]:.6f})")
print(f"Analityczny call: {price_call:.6f}  Różnica: {call_price_mc - price_call:.6f}\n")

MC call: 8.442617 ± 0.059013 (95% CI 8.383604, 8.501629)
Analityczny call: 8.433319  Różnica: 0.009298

